In [64]:
import os
import time
import pandas as pd
import re
import requests
from datetime import datetime, timedelta
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup

In [43]:
# 2. 기본 정보(한글명 + 관심수) 동시 추출 함수
def get_kream_basic_info(product_id):
    url = f"https://kream.co.kr/products/{product_id}"
    driver.get(url)
    
    time.sleep(1.5) 
    soup = BeautifulSoup(driver.page_source, 'html.parser')
    
    info = {
        "한글명": "Unknown",
        "관심수": 0
    }
    
    # --- A. 관심수 추출 ---
    target_selector = f'[data-sdui-id="product_wish_count/{product_id}"]'
    wish_element = soup.select_one(target_selector)
    if not wish_element: wish_element = soup.select_one('[data-sdui-id*="product_wish_count"]')
    
    if wish_element:
        wish_str = wish_element.get_text(strip=True)
        if '만' in wish_str:
            num_str = re.sub(r'[^0-9.]', '', wish_str)
            info["관심수"] = int(float(num_str) * 10000) if num_str else 0
        else:
            num_str = re.sub(r'[^0-9]', '', wish_str)
            info["관심수"] = int(num_str) if num_str else 0

    # --- B. 제품명(한글) 추출 ---
    p_tags = soup.find_all('p')
    for p in p_tags:
        style = p.get('style', '')
        if not style: continue
        
        # 한국어 상품명 (글자크기 15 & 한줄 제한)을 찾으면 바로 저장 후 탐색 종료
        if 'font-size:15' in style and 'line-clamp:1' in style:
            info["한글명"] = p.get_text(strip=True)
            break
                
    return info

In [34]:
"""
test_ids = ["282500"]

print("데이터 추출을 시작합니다!\n" + "-"*30)

for pid in test_ids:
    wish = get_kream_wish(pid)
    print(f"👉 상품 [{pid}] 관심수: {wish:,} 개")

print("-" * 30 + "\n✅ 추출 완료!")
"""

'\ntest_ids = ["282500"]\n\nprint("데이터 추출을 시작합니다!\n" + "-"*30)\n\nfor pid in test_ids:\n    wish = get_kream_wish(pid)\n    print(f"👉 상품 [{pid}] 관심수: {wish:,} 개")\n\nprint("-" * 30 + "\n✅ 추출 완료!")\n'

In [71]:
# ==========================================
# 🌟 함수 2. 상세 정보 추출
# ==========================================
def get_kream_details(product_id):
    print(f"\n🚨 [{product_id}] '혜택 더보기' 위의 '>' 버튼을 눌러 상세 정보를 펴주세요.")
    input("👉 모델번호/발매일/색상 등이 보이면 엔터(Enter)를 치세요! : ")
    
    soup = BeautifulSoup(driver.page_source, 'html.parser')
    details = {"모델번호": None, "발매일": None, "발매가": None, "색상": None} 
    
    p_tags = soup.find_all('p')
    for p in p_tags:
        text = p.get_text(strip=True)
        
        if text.startswith("모델번호"):
            parts = text.split(maxsplit=1)
            if len(parts) > 1: 
                val = parts[1].strip()
                if "정보 없음" not in val and val != "-":
                    details["모델번호"] = val

        elif text.startswith("발매일"):
            parts = text.split(maxsplit=1)
            if len(parts) > 1: 
                val = parts[1].strip()
                match = re.search(r'\d{2}/\d{2}/\d{2}', val)
                if match:
                    details["발매일"] = match.group()
                    
        elif text.startswith("발매가"):
            parts = text.split(maxsplit=1)
            if len(parts) > 1:
                val = parts[1].strip()
                price_str = re.sub(r'[^0-9]', '', val) 
                details["발매가"] = int(price_str) if price_str else None
                
        elif text.startswith("색상"):
            parts = text.split(maxsplit=1)
            if len(parts) > 1:
                val = parts[1].strip()
                if "정보 없음" not in val and val != "-":
                    # 🌟 이 부분이 누락되어 있었습니다! 색상 정상 입력됩니다.
                    details["색상"] = val 
                    
    return details

In [45]:
"""
test_id = "282500"

print("1️⃣ 기본 정보(이름, 하트) 자동 추출 중...")
basic_info = get_kream_basic_info(test_id)
print(f" - 한글명: {basic_info['한글명']}")
print(f" - 관심수: {basic_info['관심수']:,} 개")

print("\n2️⃣ 상세 정보(발매일 등) 추출...")
# 창이 열리면 '>' 버튼을 누르고 엔터를 치세요!
details_info = get_kream_details(test_id)
print(f" - 발매일: {details_info['발매일']}")
print(f" - 발매가: {details_info['발매가']:,} 원")
print(f" - 색상: {details['색상']}")
"""

1️⃣ 기본 정보(이름, 하트) 자동 추출 중...
 - 한글명: 나이키 ACG 루퍼스 라임스톤 앤 블랙
 - 관심수: 22,000 개

2️⃣ 상세 정보(발매일 등) 추출...

🚨 [282500] 상세 정보 창을 직접 열어주세요!


👉 화면에 모델번호/발매일 등이 보이면 엔터(Enter)를 치세요! :  


 - 발매일: 24/05/10
 - 발매가: 129,000 원
 - 색상: Limestone/Limestone/Black/Black


In [72]:
# ==========================================
# 🌟 함수 3. 거래 내역 추출 (발매일 없으면 '첫 거래일' 기준)
# ==========================================
def get_kream_transactions_1month(product_id, release_dt_str, is_womens):
    release_dt = None
    if release_dt_str:
        try:
            release_dt = pd.to_datetime("20" + release_dt_str, format="%Y/%m/%d").date()
        except:
            pass

    print(f"\n🚨 [{product_id}] '체결 내역' 창을 열어주세요!")
    print(f"💡 정렬 버튼(↑↓)을 눌러 '과거순'이 맨 위로 오게 한 뒤 딱 한 달 치만 스크롤 하세요.")
    
    if release_dt:
        print(f"   (가이드: 명시된 발매일 [{release_dt}] 기준 한 달 치)")
    else:
        print(f"   (가이드: 발매일이 없으므로, '가장 오래된 거래일'을 기준으로 한 달 치를 긁어옵니다)")
        
    input("👉 정렬과 스크롤을 마쳤으면 엔터(Enter)를 치세요! : ")
    
    def parse_kream_date(d_str):
        d_str = str(d_str)
        if "분 전" in d_str or "시간 전" in d_str: return today_date
        elif "일 전" in d_str:
            days_ago = int(re.sub(r'[^0-9]', '', d_str))
            return today_date - timedelta(days=days_ago)
        else:
            match = re.search(r'\d{2}/\d{2}/\d{2}', d_str)
            if match: return pd.to_datetime("20" + match.group(), format="%Y/%m/%d").date()
            return None

    soup = BeautifulSoup(driver.page_source, 'html.parser')
    rows = soup.find_all('div', class_='body_list')
    
    golden_range = ['235', '240', '245'] if is_womens else ['265', '270', '275']
    transactions = []
    
    for row in rows:
        cols = row.find_all('div', class_='list_txt')
        if len(cols) >= 3:
            price_str = cols[1].get_text(strip=True)
            if "원" not in price_str: continue
            
            date_str = cols[2].get_text(strip=True)
            trade_date = parse_kream_date(date_str)
            if not trade_date: continue
                
            # 🌟 [핵심] 발매일이 없으면, 가장 처음 잡히는 날짜(맨 위 과거순 데이터)를 발매일로 고정!
            if release_dt is None:
                release_dt = trade_date
                print(f"   👉 [발매일 대체] 가장 오래된 거래일({release_dt})을 발매 기준으로 설정합니다!")

            days_since = (trade_date - release_dt).days
            
            if days_since > 30:
                print(f"   *기준일({release_dt}) 30일 초과! 조기 종료합니다. (현재: {trade_date})")
                break
                
            if days_since < 0: continue

            size_raw = cols[0].get_text(strip=True)
            match = re.search(r'\d{3}', size_raw) 
            if not match: continue
            size_num = match.group() 
            if size_num not in golden_range: continue
                
            price = int(re.sub(r'[^0-9]', '', price_str))
            transactions.append({"price": price, "days_since": days_since})
                
    df = pd.DataFrame(transactions)
    if df.empty: return {"Week1_Avg": None, "Week2_Avg": None, "Week3_Avg": None, "Week4_Avg": None}
        
    df['week_idx'] = df['days_since'] // 7
    weekly_means = df.groupby('week_idx')['price'].mean()
    
    return {
        "Week1_Avg": int(weekly_means.get(0)) if 0 in weekly_means else None,
        "Week2_Avg": int(weekly_means.get(1)) if 1 in weekly_means else None,
        "Week3_Avg": int(weekly_means.get(2)) if 2 in weekly_means else None,
        "Week4_Avg": int(weekly_means.get(3)) if 3 in weekly_means else None 
    }

In [75]:
# 1. 크롬 드라이버 셋팅 및 맥(Mac) 보안 해제
print("🌐 크롬 브라우저를 셋팅합니다...")
driver_path = ChromeDriverManager().install()
os.system(f"xattr -cr '{driver_path}'")
os.system("xattr -cr ~/.wdm") # 꼬임 방지용 추가 해제

options = webdriver.ChromeOptions()
options.add_argument("user-agent=Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")
options.add_experimental_option("detach", True)

# 브라우저 띄우기 (전역 변수로 유지)
driver = webdriver.Chrome(service=Service(driver_path), options=options)
print("✅ 브라우저 실행 완료! (이 창을 닫지 마세요)")
# 수집할 상품 ID 리스트 (원하시는 상품들로 채워주세요)
my_product_ids = [
    "250825",  # (W) 나이키 샥스 R4 블랙
    "611602", # 나이키 x 자크뮈스 문 슈 SP 오프 누아르 캐시미어
]

ml_dataset = []

print("🚀 KREAM 머신러닝 데이터 수집\n" + "="*50)

for pid in my_product_ids:
    # 1. 기본 정보 자동 크롤링
    basic_info = get_kream_basic_info(pid)
    is_w = 1 if "(W)" in basic_info["한글명"] else 0  
    
    # 2. 상세 정보 수동 엔터
    details_info = get_kream_details(pid)
    
    # 3. 거래 내역 (과거순 정렬 후 수동 스크롤)
    weekly_data = get_kream_transactions_1month(pid, details_info["발매일"], is_w)
        
    # 4. 하나의 행(Row)으로 합치기
    row_data = {
        "Product_ID": pid,
        "Name": basic_info["한글명"],
        "Is_Womens": is_w,
        "Wish_Count": basic_info["관심수"],
        "Release_Date": details_info["발매일"],
        "Retail_Price": details_info["발매가"],
        "Color": details_info["색상"],
        "Week1_Avg": weekly_data["Week1_Avg"],
        "Week2_Avg": weekly_data["Week2_Avg"],
        "Week3_Avg": weekly_data["Week3_Avg"],
        "Week4_Avg_Target": weekly_data["Week4_Avg"]
    }
    
    ml_dataset.append(row_data)
    print(f"🎉 [{pid}] 수집 및 분석 완료!\n" + "-"*50)

# 최종 데이터프레임 생성 및 CSV 저장
df_final = pd.DataFrame(ml_dataset)
df_final.to_csv("KREAM_GoldenSize_ML_Dataset.csv", index=False, encoding="utf-8-sig")

print("\n🎊 모든 데이터셋 생성이 완료되었습니다!")
print("폴더에 'KREAM_GoldenSize_ML_Dataset.csv' 파일이 생성되었습니다.")
display(df_final)

🌐 크롬 브라우저를 셋팅합니다...
✅ 브라우저 실행 완료! (이 창을 닫지 마세요)
🚀 KREAM 머신러닝 데이터 수집

🚨 [250825] '혜택 더보기' 위의 '>' 버튼을 눌러 상세 정보를 펴주세요.


👉 모델번호/발매일/색상 등이 보이면 엔터(Enter)를 치세요! :  



🚨 [250825] '체결 내역' 창을 열어주세요!
💡 정렬 버튼(↑↓)을 눌러 '과거순'이 맨 위로 오게 한 뒤 딱 한 달 치만 스크롤 하세요.
   (가이드: 발매일이 없으므로, '가장 오래된 거래일'을 기준으로 한 달 치를 긁어옵니다)


👉 정렬과 스크롤을 마쳤으면 엔터(Enter)를 치세요! :  


   👉 [발매일 대체] 가장 오래된 거래일(2024-03-14)을 발매 기준으로 설정합니다!
   *기준일(2024-03-14) 30일 초과! 조기 종료합니다. (현재: 2024-04-14)
🎉 [250825] 수집 및 분석 완료!
--------------------------------------------------

🚨 [611602] '혜택 더보기' 위의 '>' 버튼을 눌러 상세 정보를 펴주세요.


👉 모델번호/발매일/색상 등이 보이면 엔터(Enter)를 치세요! :  



🚨 [611602] '체결 내역' 창을 열어주세요!
💡 정렬 버튼(↑↓)을 눌러 '과거순'이 맨 위로 오게 한 뒤 딱 한 달 치만 스크롤 하세요.
   (가이드: 명시된 발매일 [2025-09-29] 기준 한 달 치)


👉 정렬과 스크롤을 마쳤으면 엔터(Enter)를 치세요! :  


   *기준일(2025-09-29) 30일 초과! 조기 종료합니다. (현재: 2025-10-30)
🎉 [611602] 수집 및 분석 완료!
--------------------------------------------------

🎊 모든 데이터셋 생성이 완료되었습니다!
폴더에 'KREAM_GoldenSize_ML_Dataset.csv' 파일이 생성되었습니다.


,Product_ID,Name,Is_Womens,Wish_Count,Release_Date,Retail_Price,Color,Week1_Avg,Week2_Avg,Week3_Avg,Week4_Avg_Target
0,250825,(W) 나이키 샥스 R4 블랙,1,36000,NaN,179000,Black/Black/Max Orange,223772,226882,237500,247476
1,611602,나이키 x 자크뮈스 문 슈 SP 오프 누아르 캐시미어,0,13000,25/09/29,239000,Off Noir/Cashmere/Gum Light Brown/Neptune Gree...,470129,430700,383553,433814
